# Notebook 03c — Hyperparameter Tuning & Feature Ablation

**Question:** How much can Optuna improve our best model+strategy combos?
And which feature groups actually carry signal?

We tune XGBoost and LightGBM on 3-class and binary (the winning strategies from nb03a/b).

**Inputs:** `artifacts/` from nb02
**Outputs:** `results/ml_tuned.csv`, `artifacts/ml_best_params.pkl`


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time, os, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")
SEED = 42; np.random.seed(SEED)
os.makedirs('results', exist_ok=True)
print("Imports OK")


## 0. Load Data

In [ ]:
X_train_3c = joblib.load('../artifacts/X_train_3c.pkl').astype(np.float32)
X_val_3c   = joblib.load('../artifacts/X_val_3c.pkl').astype(np.float32)
y_train_3c = joblib.load('../artifacts/y_train_3c.pkl')
y_val_3c   = joblib.load('../artifacts/y_val_3c.pkl')

X_train_bin = joblib.load('../artifacts/X_train_bin.pkl').astype(np.float32)
X_val_bin   = joblib.load('../artifacts/X_val_bin.pkl').astype(np.float32)
y_train_bin = joblib.load('../artifacts/y_train_bin.pkl')
y_val_bin   = joblib.load('../artifacts/y_val_bin.pkl')

feature_names  = joblib.load('../artifacts/feature_names.pkl')
feature_groups = joblib.load('../artifacts/feature_groups.pkl')

X_3c = np.vstack([X_train_3c, X_val_3c]); y_3c = np.concatenate([y_train_3c, y_val_3c])
X_bin = np.vstack([X_train_bin, X_val_bin]); y_bin = np.concatenate([y_train_bin, y_val_bin])

print(f'3-class: {X_3c.shape}')
print(f'Binary:  {X_bin.shape}')

SCOREBOARD = []
def log_exp(model, strategy, cv_mean, cv_std, n_train, notes=""):
    SCOREBOARD.append(dict(phase="tuning", model=model, strategy=strategy,
        n_features=len(feature_names), cv_f1_mean=round(cv_mean,4),
        cv_f1_std=round(cv_std,4), n_train=n_train, notes=notes))
    print(f"  {model:22s} | {strategy:8s} | F1={cv_mean:.4f} ± {cv_std:.4f} | {notes}")


## 1. Optuna Tuning — XGBoost

We use 3-fold CV inside Optuna (for speed) and 20 trials per configuration.


In [ ]:
cv_tune = StratifiedKFold(3, shuffle=True, random_state=SEED)
best_params = {}

# ── XGBoost on 3-class ──
def obj_xgb_3c(trial):
    p = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 400, step=50),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 2.0, log=True),
        "random_state": SEED, "n_jobs": -1, "verbosity": 0, "eval_metric": "mlogloss",
    }
    return cross_val_score(xgb.XGBClassifier(**p), X_3c, y_3c,
                           cv=cv_tune, scoring="f1_macro", n_jobs=1).mean()

print("Tuning XGBoost on 3-class (20 trials)...")
t0 = time.time()
study = optuna.create_study(direction="maximize")
study.optimize(obj_xgb_3c, n_trials=20)
best_params['xgb_3c'] = study.best_params
log_exp("XGBoost-tuned", "3class", study.best_value, 0, len(X_3c), f"20t, {time.time()-t0:.0f}s")

# ── XGBoost on binary ──
def obj_xgb_bin(trial):
    p = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 400, step=50),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 2.0, log=True),
        "random_state": SEED, "n_jobs": -1, "verbosity": 0, "eval_metric": "logloss",
    }
    return cross_val_score(xgb.XGBClassifier(**p), X_bin, y_bin,
                           cv=cv_tune, scoring="f1_macro", n_jobs=1).mean()

print("\nTuning XGBoost on binary (20 trials)...")
t0 = time.time()
study = optuna.create_study(direction="maximize")
study.optimize(obj_xgb_bin, n_trials=20)
best_params['xgb_bin'] = study.best_params
log_exp("XGBoost-tuned", "binary", study.best_value, 0, len(X_bin), f"20t, {time.time()-t0:.0f}s")


## 2. Optuna Tuning — LightGBM

In [ ]:
# ── LightGBM on 3-class ──
def obj_lgb_3c(trial):
    p = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 400, step=50),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 5),
        "class_weight": "balanced",
        "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    return cross_val_score(lgb.LGBMClassifier(**p), X_3c, y_3c,
                           cv=cv_tune, scoring="f1_macro", n_jobs=1).mean()

print("Tuning LightGBM on 3-class (20 trials)...")
t0 = time.time()
study = optuna.create_study(direction="maximize")
study.optimize(obj_lgb_3c, n_trials=20)
best_params['lgb_3c'] = study.best_params
log_exp("LightGBM-tuned", "3class", study.best_value, 0, len(X_3c), f"20t, {time.time()-t0:.0f}s")

# ── LightGBM on binary ──
def obj_lgb_bin(trial):
    p = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 400, step=50),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 5),
        "class_weight": "balanced",
        "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    return cross_val_score(lgb.LGBMClassifier(**p), X_bin, y_bin,
                           cv=cv_tune, scoring="f1_macro", n_jobs=1).mean()

print("\nTuning LightGBM on binary (20 trials)...")
t0 = time.time()
study = optuna.create_study(direction="maximize")
study.optimize(obj_lgb_bin, n_trials=20)
best_params['lgb_bin'] = study.best_params
log_exp("LightGBM-tuned", "binary", study.best_value, 0, len(X_bin), f"20t, {time.time()-t0:.0f}s")

print("\nBest params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")


## 3. Feature Ablation Study

Using tuned XGBoost on 3-class, we drop one feature group at a time
and measure the impact. This tells us which groups carry real signal.


In [ ]:
print("=== Feature Ablation (tuned XGBoost, 3-class, 3-fold CV) ===")

p = best_params['xgb_3c']
def make_tuned_xgb():
    return xgb.XGBClassifier(
        max_depth=p["max_depth"], learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"], subsample=p["subsample"],
        colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
        reg_lambda=p["reg_lambda"],
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss")

cv_abl = StratifiedKFold(3, shuffle=True, random_state=SEED)

# Baseline
base_scores = cross_val_score(make_tuned_xgb(), X_3c, y_3c, cv=cv_abl, scoring="f1_macro", n_jobs=-1)
base_mean = base_scores.mean()
print(f"  ALL features: F1 = {base_mean:.4f} ± {base_scores.std():.4f}")

ablation = {}
for gname, gfeats in feature_groups.items():
    all_feats = list(feature_names)
    # Map feature group names to the actual column names after preprocessing
    # feature_names has format: num_X, cat_X, or binary name
    drop_idx = []
    for f in gfeats:
        for i, fn in enumerate(all_feats):
            if f in fn or fn == f:
                drop_idx.append(i)
    drop_idx = list(set(drop_idx))
    keep_idx = [i for i in range(len(all_feats)) if i not in drop_idx]

    if len(keep_idx) == len(all_feats):
        continue

    scores = cross_val_score(make_tuned_xgb(), X_3c[:, keep_idx], y_3c,
                              cv=cv_abl, scoring="f1_macro", n_jobs=-1)
    delta = scores.mean() - base_mean
    ablation[gname] = {"f1": scores.mean(), "delta": delta, "n_dropped": len(drop_idx)}
    d = f"+{delta:.4f}" if delta > 0 else f"{delta:.4f}"
    print(f"  Drop {gname:12s} ({len(drop_idx)} feats): F1={scores.mean():.4f} (delta={d})")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
groups = sorted(ablation.keys(), key=lambda x: ablation[x]["delta"])
deltas = [ablation[g]["delta"] for g in groups]
colors = ["#1D9E75" if d >= 0 else "#E24B4A" for d in deltas]
ax.barh(groups, deltas, color=colors, edgecolor="white", height=0.5)
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlabel("Delta Macro F1 (vs all features)")
ax.set_title("Feature Group Importance (drop-one-group)")
for i, (g, d) in enumerate(zip(groups, deltas)):
    ax.text(d + (0.001 if d >= 0 else -0.001), i,
            f"{d:+.4f}", va="center", ha="left" if d >= 0 else "right", fontsize=10)
plt.tight_layout()
plt.savefig('results/nb03c_ablation.png', bbox_inches='tight')
plt.show()


In [ ]:
# Save everything
joblib.dump(best_params, '../artifacts/ml_best_params.pkl')
print("Saved: artifacts/ml_best_params.pkl")

sb = pd.DataFrame(SCOREBOARD)
sb.to_csv('results/ml_tuned.csv', index=False)
print(f"Saved: results/ml_tuned.csv ({len(sb)} experiments)")
print()
print(sb.to_string(index=False))
